In [1]:
!pip install numpy==1.23.5 karateclub
!pip install rdflib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of karateclub to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 52.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 96.3 MB/s eta 0:00

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 1.5 MB/s eta 0:00:00


In [2]:
!pip install node2vec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 39.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
  Attempting uninstall: networkx
    Found existing installation: networkx 2.6.3
    Uninstalling networkx-2.6.3:
      Successfully uninstalled networkx-2.6.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
karateclub 1.3.0 requires networkx<2.7, but you have networkx 3.4.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.3.5 which is incompatible.
geopandas 1.0.1 requires pandas>=1.4.0, but you have pandas 1.3.5 which is incompatible.
bigfra

In [2]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import XSD
from datetime import datetime, timedelta
from collections import defaultdict
import hashlib
import networkx as nx
from karateclub import Graph2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [29]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random
import hashlib

# Namespaces
ex = Namespace("http://example.org/")
health = Namespace("http://example.org/health/")
finance = Namespace("http://example.org/finance/")
general = Namespace("http://example.org/general/")
time = Namespace("http://example.org/time/")

g = Graph()
g.bind("ex", ex)
g.bind("health", health)
g.bind("finance", finance)
g.bind("general", general)
g.bind("time", time)

def generate_stmt_id(s, p, o):
    raw = f"{str(s)}_{str(p)}_{str(o)}"
    return URIRef(f"http://example.org/event/{hashlib.md5(raw.encode()).hexdigest()}")

def add_temporal_triple(g, s, p, o, timestamp):
    g.add((s, p, o))
    stmt_id = generate_stmt_id(s, p, o)
    g.add((stmt_id, time.timestamp, Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

# --- Simulated TKG Drift Patterns with Namespaces and GT Labels ---

def simulate_running_drift(g, person, start_date):
    # Week 1-2: Running in Park (Health context)
    for i in range(0, 14):
        d = start_date + timedelta(days=i)
        add_temporal_triple(g, person, health.participatedIn, health.Running, d)
        add_temporal_triple(g, person, health.locatedAt, health.Park, d)
        g.add((health.Park, ex.hasLocationLabel, Literal("outdoor", datatype=XSD.string)))

    # Week 3-4: Switch to Yoga at WellnessCenter (Health drift)
    for i in range(14, 30):
        d = start_date + timedelta(days=i)
        add_temporal_triple(g, person, health.participatedIn, health.Yoga, d)
        add_temporal_triple(g, person, health.locatedAt, health.WellnessCenter, d)
        g.add((health.WellnessCenter, ex.hasLocationLabel, Literal("indoor", datatype=XSD.string)))

def simulate_trading_growth(g, person, start_date):
    # Week 1-2: No trading

    # Week 3-4: Trading begins and grows
    for i in range(14, 30):
        d = start_date + timedelta(days=i)
        add_temporal_triple(g, person, finance.investedIn, finance.Stocks, d)
        add_temporal_triple(g, person, finance.accountAt, finance.BankA, d)
        add_temporal_triple(g, person, finance.transactionAmount,
                            Literal(random.randint(100, 500), datatype=XSD.float), d)
        g.add((finance.BankA, ex.hasLocationLabel, Literal("digital", datatype=XSD.string)))

# --- Generate Graph ---

person = ex.JohnDoe
start_date = datetime(2024, 1, 1)

simulate_running_drift(g, person, start_date)
simulate_trading_growth(g, person, start_date)

# Return total triple count for verification
len(g)


132

In [51]:
from collections import defaultdict

# Initialize GT dictionary: person URI -> {'type': ..., 'start': ...}
gt_drift_labels = {}

NUM_PEOPLE = 50
start_date = datetime(2024, 1, 1)
people = [ex[f"Person{i}"] for i in range(NUM_PEOPLE)]
random.shuffle(people)

for person in people:
    # Random drift start date
    offset_days = random.randint(0, 30)
    drift_start = start_date + timedelta(days=offset_days)

    # Choose drift type
    drift_type = random.choice(["health", "finance"])

    # Store ground truth info
    gt_drift_labels[person] = {
        "type": drift_type,
        "start": drift_start
    }

    # Simulate drift
    if drift_type == "health":
        simulate_running_drift(g, person, drift_start)
    else:
        simulate_trading_growth(g, person, drift_start)

    # Background type triple
    g.add((person, RDF.type, ex.Person))

# Display sample GT
for i, (person, info) in enumerate(gt_drift_labels.items()):
    print(f"{person.split('/')[-1]} → Drift: {info['type']} @ {info['start'].strftime('%Y-%m-%d')}")
    if i == 9:
        break  # print only first 10


Person3 → Drift: finance @ 2024-01-28
Person28 → Drift: health @ 2024-01-26
Person31 → Drift: finance @ 2024-01-24
Person48 → Drift: health @ 2024-01-14
Person46 → Drift: finance @ 2024-01-23
Person27 → Drift: health @ 2024-01-26
Person32 → Drift: health @ 2024-01-31
Person0 → Drift: finance @ 2024-01-05
Person13 → Drift: finance @ 2024-01-12
Person7 → Drift: finance @ 2024-01-17


In [52]:
len(g)

6172

SIMPLE approach

In [30]:


# === Config ===
TIME_PRED = URIRef("http://example.org/time/timestamp")

# --- Utilities ---
def get_timestamp_literal(triple_graph, stmt_id):
    for _, _, ts in triple_graph.triples((stmt_id, TIME_PRED, None)):
        return datetime.fromisoformat(str(ts))
    return None

def extract_context_subgraph(graph, person_uri, context_prefix, until_time=None):
    G = nx.DiGraph()

    for s, p, o in graph:
        if not isinstance(o, URIRef):
            continue
        stmt_id = URIRef(f"http://example.org/event/{hashlib.md5(f'{s}_{p}_{o}'.encode()).hexdigest()}")
        ts = get_timestamp_literal(graph, stmt_id)
        if until_time and (not ts or ts > until_time):
            continue
        if str(s) != str(person_uri):
            continue
        if str(p).startswith(context_prefix) or str(o).startswith(context_prefix):
            G.add_edge(str(s), str(o), label=str(p).split("/")[-1])

    return G

def extract_future_snapshots(graph, person_uri, context_prefix, from_time=None, window_days=30, num_snapshots=6):
    snapshots = []
    for i in range(num_snapshots):
        until_time = from_time + timedelta(days=window_days * (i + 1))
        G = extract_context_subgraph(graph, person_uri, context_prefix, until_time=until_time)
        snapshots.append((until_time, G))
    return snapshots

# def encode_graphs(graphs):
#     relabeled_graphs = []
#     for G in graphs:
#         mapping = {node: i for i, node in enumerate(G.nodes())}
#         G_relabel = nx.relabel_nodes(G, mapping)
#         relabeled_graphs.append(G_relabel)

#     model = Graph2Vec(dimensions=64, wl_iterations=2, min_count=1, epochs=10)
#     model.fit(relabeled_graphs)
#     embeddings = model.get_embedding()
#     return embeddings

def encode_graphs(graphs):
    relabeled_graphs = []
    for G in graphs:
        if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
            print(" Skipping empty graph...")
            continue
        mapping = {node: i for i, node in enumerate(G.nodes())}
        G_relabel = nx.relabel_nodes(G, mapping)
        relabeled_graphs.append(G_relabel)

    if not relabeled_graphs:
        raise ValueError("No valid graphs to embed.")

    model = Graph2Vec(dimensions=64, wl_iterations=2, min_count=1, epochs=10)
    model.fit(relabeled_graphs)
    embeddings = model.get_embedding()
    return embeddings


def classify_evolution(reference_emb, future_embs):
    sims = [cosine_similarity([reference_emb], [emb])[0][0] for emb in future_embs]
    deltas = np.diff(sims)

    results = []
    for i, sim in enumerate(sims):
        if i == 0:
            trend = "baseline"
        elif deltas[i-1] > 0.1:
            trend = "growing"
        elif deltas[i-1] < -0.1:
            trend = "decaying"
        elif abs(deltas[i-1]) < 0.05:
            trend = "stable"
        else:
            trend = "drifting"
        results.append((i, sim, trend))
    return results

def run_context_evolution(graph, person_uri, context="health", window_days=30, num_snapshots=6):
    context_prefix = f"http://example.org/{context}/"

    all_timestamps = [get_timestamp_literal(graph, URIRef(stmt)) for stmt in graph.subjects(predicate=TIME_PRED)]
    all_timestamps = sorted([ts for ts in all_timestamps if ts])
    if not all_timestamps:
        raise ValueError("No timestamps found.")
    print (all_timestamps)

    final_time = all_timestamps[-1]
    T0 = final_time - timedelta(days=window_days * num_snapshots)

    G_ref = extract_context_subgraph(graph, person_uri, context_prefix, until_time=T0)
    future_snapshots = extract_future_snapshots(graph, person_uri, context_prefix, from_time=T0, window_days=window_days, num_snapshots=num_snapshots)

    graphs = [G_ref] + [G for _, G in future_snapshots]
    for i, G in enumerate(graphs):
        print(f"Snapshot {i}: Nodes={G.number_of_nodes()}, Edges={G.number_of_edges()}")
    # Inside run_context_evolution before encode_graphs
    if G_ref.number_of_nodes() == 0:
        print("⚠️ Reference subgraph is empty! Consider changing 'window_days' or data simulation.")
        adsad

    embeddings = encode_graphs(graphs)

    evolution = classify_evolution(embeddings[0], embeddings[1:])

    print(f"Subgraph evolution for: {person_uri.split('/')[-1]} (context: {context})")
    for i, sim, label in evolution:
        print(f" T{i+1}: similarity = {sim:.3f}, label = {label}")

    return evolution


In [35]:
run_context_evolution(g, ex.JohnDoe, context="finance", window_days=10, num_snapshots=2)

[datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 1, 0, 0), 

NameError: name 'adsad' is not defined

One person

In [45]:
from rdflib import URIRef, Literal
from rdflib.namespace import XSD
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime, timedelta
from collections import defaultdict
from gensim.models import Word2Vec
import numpy as np

def extract_weekly_subgraphs(g, anchor, window_days=7):
    """
    Extracts 2-hop subgraphs weekly around a given anchor node.
    """
    # Collect all timestamped events
    time_stamped = []
    for stmt in g.subjects(predicate=URIRef("http://example.org/time/timestamp")):
        ts = g.value(stmt, URIRef("http://example.org/time/timestamp"))
        if isinstance(ts, Literal):
            dt = datetime.fromisoformat(str(ts))
            time_stamped.append((stmt, dt))

    # Partition events by week
    time_stamped.sort(key=lambda x: x[1])
    min_time = time_stamped[0][1]
    week_buckets = defaultdict(list)

    for stmt, dt in time_stamped:
        week_idx = (dt - min_time).days // window_days
        week_buckets[week_idx].append(stmt)

    # Build 2-hop subgraphs for each week
    snapshots = []
    for week in sorted(week_buckets.keys()):
        G = nx.MultiDiGraph()
        stmts = week_buckets[week]

        for stmt in stmts:
            for s, p, o in g:
                sid = URIRef(f"http://example.org/event/{hash_triple(s, p, o)}")
                if sid == stmt or (isinstance(o, URIRef) and sid in g.subjects(predicate=URIRef("http://example.org/time/timestamp"))):
                    G.add_edge(str(s), str(o), label=str(p))

        # Add 2-hop neighbors of anchor
        if str(anchor) in G:
            neighbors_1 = set(G.successors(str(anchor))) | set(G.predecessors(str(anchor)))
            neighbors_2 = set()
            for n1 in neighbors_1:
                neighbors_2 |= set(G.successors(n1)) | set(G.predecessors(n1))
            sub_nodes = {str(anchor)} | neighbors_1 | neighbors_2
            SG = G.subgraph(sub_nodes).copy()
        else:
            SG = nx.MultiDiGraph()  # empty

        snapshots.append(SG)
    return snapshots

def hash_triple(s, p, o):
    return hashlib.md5(f"{s}_{p}_{o}".encode()).hexdigest()

def train_node2vec_embeddings(graphs, dimensions=64, window_size=3, walk_length=5, workers=2):
    """
    Train Node2Vec on list of NetworkX graphs (snapshots).
    Returns a list of mean node embeddings for each graph.
    """
    embeddings = []
    for G in graphs:
        if len(G.nodes) < 2:
            embeddings.append(np.zeros(dimensions))
            continue

        walks = []
        for node in G.nodes():
            walk = random_walk(G, node, walk_length)
            walks.append(walk)

        model = Word2Vec(sentences=walks, vector_size=dimensions, window=window_size, min_count=1, sg=1, workers=workers, epochs=20)
        node_vecs = np.array([model.wv[n] for n in model.wv.index_to_key])
        mean_vec = node_vecs.mean(axis=0)
        embeddings.append(mean_vec)

    return embeddings

def random_walk(G, start_node, length):
    walk = [start_node]
    while len(walk) < length:
        cur = walk[-1]
        neighbors = list(G.successors(cur)) + list(G.predecessors(cur))
        if not neighbors:
            break
        walk.append(random.choice(neighbors))
    return walk

def detect_drift(embeddings, threshold=0.1):
    """
    Computes cosine similarity between consecutive weeks.
    Returns list of drift scores and labels (1=drift, 0=no drift).
    """
    drifts = []
    for i in range(1, len(embeddings)):
        sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
        drift = 1 if sim < (1 - threshold) else 0
        drifts.append((i, 1 - sim, drift))
    return drifts

# === Full Pipeline ===

def run_node2vec_drift_pipeline(g, anchor, window_days=7):
    snapshots = extract_weekly_subgraphs(g, anchor, window_days=window_days)
    print(f"Extracted {len(snapshots)} weekly snapshots.")
    embeddings = train_node2vec_embeddings(snapshots)
    drift_results = detect_drift(embeddings)
    print("Drift Detection Results (WeekIndex, 1 - CosineSim, DriftLabel):")
    for r in drift_results:
        print(r)
    return drift_results


In [46]:
drift_results = run_node2vec_drift_pipeline(g, ex.JohnDoe, window_days=7)


Extracted 4 weekly snapshots.
Drift Detection Results (WeekIndex, 1 - CosineSim, DriftLabel):
(1, 0.2804855704307556, 1)
(2, 0.031193792819976807, 0)
(3, 0.14645683765411377, 1)


In [47]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_drift_predictions(drift_results, ground_truth):
    """
    Compare predicted drifts with GT labels.
    Returns accuracy, precision, recall, F1.
    """
    y_true = []
    y_pred = []

    for week_idx, _, pred_label in drift_results:
        if week_idx in ground_truth:
            y_true.append(ground_truth[week_idx])
            y_pred.append(pred_label)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Accuracy:  {acc:.2f}")
    print(f"Precision: {prec:.2f}")
    print(f"Recall:    {rec:.2f}")
    print(f"F1 Score:  {f1:.2f}")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_drift_predictions(drift_results, ground_truth):
    """
    Compare predicted drifts with GT labels.
    Returns accuracy, precision, recall, F1.
    """
    y_true = []
    y_pred = []

    for week_idx, _, pred_label in drift_results:
        if week_idx in ground_truth:
            y_true.append(ground_truth[week_idx])
            y_pred.append(pred_label)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Accuracy:  {acc:.2f}")
    print(f"Precision: {prec:.2f}")
    print(f"Recall:    {rec:.2f}")
    print(f"F1 Score:  {f1:.2f}")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }


In [48]:
# Ground truth: Week 1-2 normal, Week 3+ has drift (e.g. Running stops, Trading starts)
gt = {
    1: 0,  # no drift
    2: 0,  # drift (Running→Yoga)
    3: 1,  # drift (Trading starts)
    4: 0   # continues changing behavior
}

# Run after Node2Vec pipeline
drift_results = run_node2vec_drift_pipeline(g, ex.JohnDoe, window_days=7)
evaluate_drift_predictions(drift_results, gt)


Extracted 4 weekly snapshots.
Drift Detection Results (WeekIndex, 1 - CosineSim, DriftLabel):
(1, 0.2849937081336975, 1)
(2, 0.030498266220092773, 0)
(3, 0.14602446556091309, 1)
Accuracy:  0.67
Precision: 0.50
Recall:    1.00
F1 Score:  0.67


{'accuracy': 0.6666666666666666,
 'precision': 0.5,
 'recall': 1.0,
 'f1': 0.6666666666666666}

N person

In [ ]:
# Re-import necessary libraries and prepare the pipeline to run for all 50 people

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import numpy as np
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Helper functions for drift detection
def hash_triple(s, p, o):
    return hashlib.md5(f"{s}_{p}_{o}".encode()).hexdigest()

def extract_weekly_subgraphs(g, anchor, window_days=7):
    time_stamped = []
    for stmt in g.subjects(predicate=time.timestamp):
        ts = g.value(stmt, time.timestamp)
        if isinstance(ts, Literal):
            dt = datetime.fromisoformat(str(ts))
            time_stamped.append((stmt, dt))

    time_stamped.sort(key=lambda x: x[1])
    min_time = time_stamped[0][1]
    week_buckets = defaultdict(list)

    for stmt, dt in time_stamped:
        week_idx = (dt - min_time).days // window_days
        week_buckets[week_idx].append(stmt)

    snapshots = []
    for week in sorted(week_buckets.keys()):
        G = nx.MultiDiGraph()
        stmts = week_buckets[week]
        for stmt in stmts:
            for s, p, o in g:
                sid = URIRef(f"http://example.org/event/{hash_triple(s, p, o)}")
                if sid == stmt:
                    G.add_edge(str(s), str(o), label=str(p))

        if str(anchor) in G:
            neighbors_1 = set(G.successors(str(anchor))) | set(G.predecessors(str(anchor)))
            neighbors_2 = set()
            for n1 in neighbors_1:
                neighbors_2 |= set(G.successors(n1)) | set(G.predecessors(n1))
            sub_nodes = {str(anchor)} | neighbors_1 | neighbors_2
            SG = G.subgraph(sub_nodes).copy()
        else:
            SG = nx.MultiDiGraph()
        snapshots.append(SG)
    return snapshots

def random_walk(G, start_node, length):
    walk = [start_node]
    while len(walk) < length:
        cur = walk[-1]
        neighbors = list(G.successors(cur)) + list(G.predecessors(cur))
        if not neighbors:
            break
        walk.append(random.choice(neighbors))
    return walk

def train_node2vec_embeddings(graphs, dimensions=64, window_size=3, walk_length=5, workers=2):
    embeddings = []
    for G in graphs:
        if len(G.nodes) < 2:
            embeddings.append(np.zeros(dimensions))
            continue
        walks = [random_walk(G, node, walk_length) for node in G.nodes()]
        model = Word2Vec(sentences=walks, vector_size=dimensions, window=window_size, min_count=1, sg=1, workers=workers, epochs=20)
        node_vecs = np.array([model.wv[n] for n in model.wv.index_to_key])
        mean_vec = node_vecs.mean(axis=0)
        embeddings.append(mean_vec)
    return embeddings

def detect_drift(embeddings, threshold=0.1):
    drifts = []
    for i in range(1, len(embeddings)):
        sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
        drift = 1 if sim < (1 - threshold) else 0
        drifts.append((i, 1 - sim, drift))
    return drifts

def compute_gt_week_index(gt_start_date, global_start_date, window_days=7):
    return (gt_start_date - global_start_date).days // window_days

# Run the pipeline for all people
global_start = datetime(2024, 1, 1)
y_true_all = []
y_pred_all = []

for person in people:
    snapshots = extract_weekly_subgraphs(g, person)
    embeddings = train_node2vec_embeddings(snapshots)
    drift_results = detect_drift(embeddings)
    gt_week = compute_gt_week_index(gt_drift_labels[person]["start"], global_start)
    for week_idx, _, pred_label in drift_results:
        if week_idx == gt_week or week_idx == gt_week + 1:
            y_true_all.append(1)
        else:
            y_true_all.append(0)
        y_pred_all.append(pred_label)



In [ ]:
# Evaluate performance
accuracy = accuracy_score(y_true_all, y_pred_all)
precision = precision_score(y_true_all, y_pred_all, zero_division=0)
recall = recall_score(y_true_all, y_pred_all, zero_division=0)
f1 = f1_score(y_true_all, y_pred_all, zero_division=0)

accuracy, precision, recall, f1


EXTRA code we used for drift detection

In [ ]:
# Detect drift in health domain for JohnDoe
run_context_evolution(graph=g, person_uri=ex.JohnDoe, context="finance")


In [19]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random

# --- Namespaces ---
general = Namespace("http://example.org/personal/general/")
health1 = Namespace("http://example.org/personal/health/")
location = Namespace("http://example.org/personal/location/")
person1 = Namespace("http://example.org/personal/person/")
travell = Namespace("http://example.org/personal/travel/")
time = Namespace("http://example.org/personal/time/")

# --- Initialize Graph ---
g = Graph()
g.bind("general", general)
g.bind("health1", health1)
g.bind("location", location)
g.bind("person1", person1)
g.bind("travell", travell)
g.bind("time", time)

# --- Parameters ---
person = person1.Person
days = 365
start_date = datetime(2023, 1, 1)
drift_day = 140  # health drops due to travel (around week 20)
recover_day = 196  # health returns to normal (around week 28)

# --- Helper Function ---
def add_literal_with_time(subject, predicate, value, timestamp):
    value_literal = Literal(value)
    g.add((subject, predicate, value_literal))
    g.add((value_literal, time.timestamp, Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

# --- Simulate ---
for day in range(days):
    ts = start_date + timedelta(days=day)

    # Create health activity nodes per day
    sleep_node = URIRef(f"{health1}Sleep_{day}")
    hr_node = URIRef(f"{health1}HeartRate_{day}")
    steps_node = URIRef(f"{health1}Steps_{day}")

    # Link to person
    g.add((person, health1.hasHealthData, sleep_node))
    g.add((person, health1.hasHealthData, hr_node))
    g.add((person, health1.hasHealthData, steps_node))

    # Add travel events
    if day == drift_day:
        trip = URIRef(f"{travell}Trip_{day}")
        city = location.CityB
        g.add((person, travell.travelTo, city))
        g.add((person, travell.hasTravelBooking, trip))
        add_literal_with_time(trip, travell.placeName, "CityB", ts)

    if day == recover_day:
        return_trip = URIRef(f"{travell}TripBack_{day}")
        g.add((person, travell.travelTo, location.CityA))
        g.add((person, travell.hasTravelBooking, return_trip))
        add_literal_with_time(return_trip, travell.placeName, "CityA", ts)

    # Health values
    if day == drift_day:
        duration = random.randint(120, 200)
        heart = random.randint(95, 110)
        steps = random.randint(1000, 2000)
    elif day == recover_day:
        duration = random.randint(400, 500)
        heart = random.randint(60, 75)
        steps = random.randint(7000, 10000)
    else:
        duration = random.randint(350, 500)
        heart = random.randint(65, 80)
        steps = random.randint(5000, 9000)

    # Link health info with values and timestamp
    add_literal_with_time(sleep_node, health1.hasDuration, duration, ts)
    add_literal_with_time(hr_node, health1.hasHeartRate, heart, ts)
    add_literal_with_time(steps_node, health1.hasSteps, steps, ts)

print(f"Drift injected on day {drift_day} (travel) and recovery on day {recover_day}.")


Drift injected on day 140 (travel) and recovery on day 196.


In [20]:
# Required libraries
from rdflib import Graph, URIRef, Namespace, Literal
from rdflib.namespace import XSD
from datetime import datetime, timedelta
from collections import defaultdict
import networkx as nx
from node2vec import Node2Vec
from karateclub import Graph2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Namespaces
# general = Namespace("http://example.org/personal/general/")
# health1 = Namespace("http://example.org/personal/health/")
# person1 = Namespace("http://example.org/personal/person/")
# travell = Namespace("http://example.org/personal/travel/")
# time = Namespace("http://example.org/personal/time/")

# # Load Graph
# g = Graph()
# g.parse("simulated_health_travel_drift_daily.ttl", format="ttl")

# --- 1. Snapshot Extraction (Weekly Buckets) ---
def extract_weekly_snapshot(graph, start_date, target_week):
    grouped = defaultdict(list)
    for o, p, t in graph.triples((None, time.timestamp, None)):
        if isinstance(t, Literal):
            dt = datetime.fromisoformat(str(t))
            week_num = (dt - start_date).days // 7
            grouped[week_num].append(str(o))

    snapshot_triples = []
    for s, p, o in graph:
        if str(o) in grouped[target_week] or str(s) in grouped[target_week]:
            snapshot_triples.append((s, p, o))
    return snapshot_triples



In [21]:
# --- 2. Extract Subgraphs by Context ---
def extract_subgraphs_by_context(triples, context_prefix):
    G = nx.DiGraph()
    for s, p, o in triples:
        if context_prefix in str(p) or context_prefix in str(o):
            G.add_edge(str(s), str(o), label=str(p).split("/")[-1])

    subgraphs = []
    for component in nx.weakly_connected_components(G):
        sg = G.subgraph(component).copy()
        if sg.number_of_edges() > 0:
            subgraphs.append(sg)
    return subgraphs

# --- 3. Compute Embeddings ---
def get_graph_embeddings(subgraphs):
    g2v = Graph2Vec(dimensions=64, wl_iterations=2, epochs=10)
    g2v.fit(subgraphs)
    graph_embeds = g2v.get_embedding()
    return graph_embeds

def get_node_embeddings(subgraph):
    if subgraph.number_of_nodes() < 2:
        return None
    n2v = Node2Vec(subgraph, dimensions=64, walk_length=5, num_walks=10, workers=1)
    model = n2v.fit(window=3, min_count=1, batch_words=4)
    emb = np.mean([model.wv[n] for n in subgraph.nodes if n in model.wv], axis=0)
    return emb

# --- 5. Compute Evolution Signature ---
def compute_evolution_signature(ref_subgraphs, fut_subgraphs):
    Sg = cosine_similarity(get_graph_embeddings(ref_subgraphs), get_graph_embeddings(fut_subgraphs))
    Sn = []
    for gi in ref_subgraphs:
        row = []
        vi = get_node_embeddings(gi)
        for gj in fut_subgraphs:
            vj = get_node_embeddings(gj)
            if vi is None or vj is None:
                row.append(0.0)
            else:
                row.append(cosine_similarity([vi], [vj])[0][0])
        Sn.append(row)
    Sn = np.array(Sn)

    # Decision
    evolution_sig = []
    for i in range(Sg.shape[0]):
        for j in range(Sg.shape[1]):
            sg_sim = Sg[i, j]
            sn_sim = Sn[i, j]
            if sg_sim > 0.7 and sn_sim > 0.7:
                label = "static"
            elif sg_sim > 0.7 and sn_sim < 0.5:
                label = "growing"
            elif sg_sim < 0.5 and sn_sim > 0.7:
                label = "possible drift"
            else:
                label = "drifted"
            evolution_sig.append((i, j, label))
    return evolution_sig, Sg, Sn



In [22]:
# --- 6. Run Full Pipeline ---
start_date = datetime(2023, 1, 1)
t1 = 0  # week 0
t2 = 20  # week 20

triples_t1 = extract_weekly_snapshot(g, start_date, t1)
triples_t2 = extract_weekly_snapshot(g, start_date, t2)

subgraphs_t1 = extract_subgraphs_by_context(triples_t1, "health")
subgraphs_t2 = extract_subgraphs_by_context(triples_t2, "health")



In [23]:
print(f"\nSubgraphs in health at time t1 (week {t1}):")
for idx, sg in enumerate(subgraphs_t2):
    print(f"--- Subgraph {idx} ---")
    for u, v, data in sg.edges(data=True):
        print(f"  ({u}) --[{data['label']}]--> ({v})")



Subgraphs in health at time t1 (week 0):
--- Subgraph 0 ---
  (http://example.org/personal/health/HeartRate_183) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_99) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_259) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_309) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_347) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_126) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_175) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_55) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_195) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_340) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_247) --[hasHeartRate]--> (80)
  (http://example.org/personal/health/HeartRate_90) --[hasHeartRate]--> (80)
  (htt

In [25]:
evol_sig, sg_mat, sn_mat = compute_evolution_signature(subgraphs_t1[0], subgraphs_t2[0])

print("Evolution Signatures between t1 and t2:")
for (i, j, label) in evol_sig:
    print(f"Subgraph {i} → {j}: {label}")

# Check if any drift
drift_detected = any(label == "drifted" or label == "possible drift" for (_, _, label) in evol_sig)
print("\nDrift Detected:" if drift_detected else "\nNo Drift Detected")


AttributeError: 'str' object has no attribute 'number_of_nodes'